In [26]:
import json
import os
from datasets import Dataset


In [27]:
import os

from jsonlines import jsonlines

# # 遍历所有 annotated 下的 jsonl 文件
# root_dir = "../datasets/annotated"
# output_dir = "./work/work"
# os.makedirs(output_dir, exist_ok=True)
# all_formatted = []
# sid_data = []
# for subdir, _, files in os.walk(root_dir):
#     for file in files:
#         if file.endswith(".jsonl"):
#             file_path = os.path.join(subdir, file)
#             print(f"Processing: {file_path}")
#             with jsonlines.open(file_path, "r") as f:
#                 sid_data = [line for line in f]

with jsonlines.open(
    "../datasets/annotated/GPT-4o/GPT-4o-interdis_topic_1.jsonl", "r"
) as f:
    sid_data = [line for line in f]


In [28]:
import pprint


pprint.pprint(sid_data[:2])  # 打印前两个样本进行检查

[{'annotations': [{'cognitive_level': '理解',
                   'discipline': '生物',
                   'discipline_transfer': '否',
                   'speaker': '学生',
                   'student_cognition_state': '理解',
                   'teacher_guidance_level': '',
                   'teacher_intent': '',
                   'teaching_strategy': '',
                   'utterance': '在学习植物分类时，我们如何区分裸子植物和被子植物？'},
                  {'cognitive_level': '分析',
                   'discipline': '生物',
                   'discipline_transfer': '否',
                   'speaker': '教师',
                   'student_cognition_state': '',
                   'teacher_guidance_level': 'L3',
                   'teacher_intent': '引导推理',
                   'teaching_strategy': '引导推理',
                   'utterance': '植物的分类不仅涉及形态特征，还与它们的生殖方式密切相关。那么，你认为裸子植物和被子植物的种子在结构上可能有什么关键区别？'},
                  {'cognitive_level': '理解',
                   'discipline': '生物',
                   'discipline_transfer': '否',

In [29]:
import hashlib


def get_content_hash(messages):
    """
    计算消息列表的 MD5 哈希指纹。
    """
    # 将列表序列化为字符串，ensure_ascii=False 保证中文一致性
    serialized = json.dumps(messages, sort_keys=True, ensure_ascii=False)
    # 计算 MD5 并截取前 8 位
    return hashlib.md5(serialized.encode("utf-8")).hexdigest()[:8]


In [30]:
collab_data = []

for entry in sid_data:
    dialogue = entry.get("dialogue", [])
    base_id = entry.get("student_id", "unknown")

    # 提取基础元数据
    single_turn_prompt = dialogue[0]["content"] if dialogue else ""
    metadata = {
        "source": "SID",
        "student_type": entry.get("student_type"),
        "topic": entry.get("topic_text", ""),
    }

    history = []
    # 用于记录已生成的 ID，防止万一有完全重复的数据行
    seen_ids = set()
    # 遍历对话，为每一个“老师”的回复生成一个独立的训练样本
    for i, turn in enumerate(dialogue):
        role = "user" if turn["role"] == "学生" else "assistant"
        content = turn["content"]

        # 如果当前是 Teacher (Assistant)，构建一个训练样本
        if role == "assistant":
            if history:  # 必须有历史才有上下文
                prompt_hash = get_content_hash(history)
                # --- 关键修改：Unique ID ---
                # 使用 {student_id}_turn_{index} 作为 conv_id
                # 欺骗 MultiturnDataset，让它认为这都是不同的对话，从而保留所有数据
                # 更稳健的 ID 生成方式，防止不同对话 ID 冲突
                unique_conv_id = f"{base_id}_turn_{i}_{prompt_hash}"
                if unique_conv_id in seen_ids:
                    continue
                seen_ids.add(unique_conv_id)
                # 构建 Style 2 格式的样本
                sample = {
                    "conv_id": unique_conv_id,
                    "single_turn_prompt": single_turn_prompt,
                    "single_turn_completion": "",  # SFT 不强制要求此字段有值，留空即可
                    "single_turn_metadata": metadata,
                    "turns": [
                        {
                            "prompt": list(history),  # 复制当前历史作为输入
                            "responses": [
                                {
                                    "completion": content,  # 老师的当前回答作为目标
                                    "score": 1.0,  # SFT 默认给满分
                                }
                            ],
                        }
                    ],
                }
                collab_data.append(sample)

        # 将当前对话加入历史，供下一轮使用
        history.append({"role": role, "content": content})

output_path = "../datasets/sid_collabllm/sid_collabllm_sft.json"
# 保存为 JSON 文件
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(collab_data, f, ensure_ascii=False, indent=2)

print(f"转换完成！")
print(f"原始 SID 对话数: {len(sid_data)}")
print(f"生成 CollabLLM 训练样本数 (切片后): {len(collab_data)}")
print(f"文件已保存至: {output_path}")

转换完成！
原始 SID 对话数: 40
生成 CollabLLM 训练样本数 (切片后): 238
文件已保存至: ../datasets/sid_collabllm/sid_collabllm_sft.json
